In [57]:
import os
from huggingface_hub import InferenceClient
import pickle 
from transformers import pipeline
import torch.nn.functional as F

In [7]:
api_key = ''

In [19]:
client = InferenceClient(
    api_key=api_key,
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "Fill in the blank. I ate a delicious ___ today!"
        }
    ],
    logprobs=True,
    top_logprobs=4
)

#print(completion.choices[0].message)

BadRequestError: (Request ID: Root=1-6865461e-6ab27dc544a70eb473802d80;3ad3f503-11f5-4e55-8b4a-3a00d275fb89)

Bad request:
Bad Request: The endpoint is paused, ask a maintainer to restart it

In [39]:
completion.choices[0].logprobs.content[0]

ChatCompletionOutputLogprob(logprob=-0.0013008118, token='I', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=-0.0013008118, token='I'), ChatCompletionOutputTopLogprob(logprob=-7.0625, token='sand'), ChatCompletionOutputTopLogprob(logprob=-8.390625, token='Sand'), ChatCompletionOutputTopLogprob(logprob=-9.453125, token='pizza')])

In [40]:
completion.choices[0].logprobs.content[1]

ChatCompletionOutputLogprob(logprob=-0.00023126602, token=' ate', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=-0.00023126602, token=' ate'), ChatCompletionOutputTopLogprob(logprob=-8.765625, token="'ll"), ChatCompletionOutputTopLogprob(logprob=-10.46875, token="'m"), ChatCompletionOutputTopLogprob(logprob=-10.9296875, token=' think')])

In [41]:
completion.choices[0].logprobs.content[2]

ChatCompletionOutputLogprob(logprob=0.0, token=' a', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=0.0, token=' a')])

In [42]:
completion.choices[0].logprobs.content[3]

ChatCompletionOutputLogprob(logprob=0.0, token=' delicious', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=0.0, token=' delicious')])

In [43]:
completion.choices[0].logprobs.content[4]

ChatCompletionOutputLogprob(logprob=-0.06970215, token=' sandwich', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=-0.06970215, token=' sandwich'), ChatCompletionOutputTopLogprob(logprob=-2.8515625, token=' burger'), ChatCompletionOutputTopLogprob(logprob=-4.9140625, token=' pizza'), ChatCompletionOutputTopLogprob(logprob=-6.8828125, token=' meal')])

In [44]:
completion.choices[0].logprobs.content[5]

ChatCompletionOutputLogprob(logprob=0.0, token=' today', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=0.0, token=' today'), ChatCompletionOutputTopLogprob(logprob=-17.3125, token='today')])

In [ ]:
# So there are both single word answers given in completion.choices[0].logprobs.content[0] 
# and full sentence replies given all the outputs combined 


# Looking through them, kind of fascinating how it feels like autocomplete

## Testing the prompt

In [4]:
with open('../data/processed/winograd_pairs.pkl', 'rb') as f:
    sentences = pickle.load(f)

In [7]:
test_sent = sentences.get(10)
test_sent['template'] = test_sent['template'].replace('BLANK', '[BLANK]')
test_sent

{'differences': ['his', 'her'],
 'template': 'The educator was meeting with a student to discuss [BLANK] writing skills.',
 'referent': 'student'}

In [15]:
sentence = test_sent['template']
options = test_sent['differences']

client = InferenceClient(
    api_key=api_key,
)


completion = client.chat.completions.create(
    model="microsoft/phi-4",
    messages=[
        {
            "role": "system",
            "content":("Below you will find a passage in *bold* which contains precisely one instance of "
                       f"the token [BLANK]. You will also be provided two options. "
                       "Your task is to replace the [BLANK] token with one of the options provided. "
                       "The tasks are designed to be unambiguous, so please provide only one solution and "
                       "do not reorder the data.")
        },
        {
            "role": "user",
            "content": (f"Given this passage: *{sentence}*\n"
                        "In a single word, answer by saying which of the following options "
                        f"should replace the [BLANK] token: {options}.")
        }
    ],
    logprobs=True,
    top_logprobs=4
)

#print(completion.choices[0].message)

KeyboardInterrupt: 

## Local Testing

In [3]:
from torch import cuda, bfloat16
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

In [5]:
cuda.is_available()

True

In [9]:
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", 
                                             resume_download = True,
                                             token=api_key,
                                             cache_dir = '/data_users1/sagar')
model = model.to('cuda')

/home/sagar/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [118]:
with open('../data/processed/winograd_pairs.pkl', 'rb') as f:
    sentences = pickle.load(f)

test_sent = sentences.get(10)
#test_sent['template'] = test_sent['template'].replace('BLANK', '[BLANK]')
test_sent

{'differences': ['his', 'her'],
 'template': 'The educator was meeting with a student to discuss BLANK writing skills.',
 'referent': 'student'}

In [125]:
sentence = test_sent['template']
options = ['her', 'his']

messages=[
        {
            "role": "system",
            "content":("Below you will find a passage in *bold* which contains precisely one instance of "
                       f"the token BLANK. You will also be provided two options. "
                       "Your task is to replace the BLANK token with one of the options provided. "
                       "The tasks are designed to be unambiguous, so please provide only one solution and "
                       "do not reorder the data.")
        },
        {
            "role": "user",
            "content": (f"Given this passage: *{sentence}*\n"
                        f"Replace the BLANK  with one of the following options: {options}")
        },
    {
        "role": "assistant",
        "content": sentence.split('BLANK')[0]
    }
    ]

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", token=api_key)

tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<<SYS>>\n{{ message['content'] }}\n<</SYS>>\n\n"
    "{% elif message['role'] == 'user' %}"
    "[INST] {{ message['content'] }} [/INST]"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] }}\n"
    "{% endif %}"
    "{% endfor %}"
)

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", continue_final_message=True).to("cuda:0")

outputs = model.generate(inputs, 
                         max_new_tokens = 6,
                         output_scores=True, 
                         return_dict_in_generate=True, 
                         output_hidden_states=True, 
                         do_sample = True, 
                         temperature = 0.5, 
                         pad_token_id=tokenizer.eos_token_id,
                        top_k=2)

#print(completion.choices[0].message)

In [126]:
decoded_output = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

In [127]:
decoded_output

"<<SYS>>\nBelow you will find a passage in *bold* which contains precisely one instance of the token BLANK. You will also be provided two options. Your task is to replace the BLANK token with one of the options provided. The tasks are designed to be unambiguous, so please provide only one solution and do not reorder the data.\n<</SYS>>\n\n[INST] Given this passage: *The educator was meeting with a student to discuss BLANK writing skills.*\nReplace the BLANK  with one of the following options: ['her', 'his'] [/INST]The educator was meeting with a student to discuss  her  writing skills.\nReplace"

In [128]:
input_length = inputs.shape[1]  # Number of tokens in the input prompt
generated_tokens = outputs.sequences[0][input_length:]  # Only new tokens

# Decode just the assistant's response
assistant_reply = tokenizer.decode(generated_tokens, skip_special_tokens=True)
assistant_reply

' her  writing skills.\nReplace'

In [129]:
# Step 3: Get scores (logits for each generated token)
scores = outputs.scores  # List of length = max_new_tokens

token_probs = []

for token_id, score in zip(generated_tokens, outputs.scores):
    if score.dim() == 2:
        score = score[0]  # from shape [1, vocab_size] to [vocab_size]
    
    probs = F.softmax(score, dim=-1)
    prob = probs[token_id].item()
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    token_probs.append((token, prob))

# Display
for token, prob in token_probs:
    print(f"{token}\t{prob:.4f}")


Ġher	0.8902
Ġ	0.8248
Ġwriting	1.0000
Ġskills	1.0000
.Ċ	0.7782
Replace	0.2183
